# Chapter 1: Matplotlib — The Foundation

### 1.1 Anatomy of a Matplotlib Chart

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')     # a clean, professional default look
sns.set_palette("husl")

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
sales = [185, 192, 210, 198, 225, 240]

plt.figure(figsize=(10, 5))
plt.plot(months, sales, marker='o', linewidth=2.5, markersize=8)
plt.title("Monthly Sales Trend – H1 2025", fontsize=14, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Sales (₹ Lakhs)")
plt.tight_layout()
plt.show()

Every Matplotlib chart, no matter how simple or complex, follows the same skeleton: create a **figure** (the canvas), plot data onto it, then label and show it. `figsize=(width, height)` is in inches and directly controls how the chart looks when exported or embedded in a report — get in the habit of setting it explicitly rather than accepting the tiny default, which looks cramped in any real document. `plt.tight_layout()` automatically adjusts spacing so labels and titles don't get clipped or overlap — add it as a near-automatic last step before `plt.show()`.

**There are, confusingly, two different "styles" of writing Matplotlib code**, and you'll see both in the wild:

In [ ]:
# Style 1: pyplot (stateful) — quick and terse, shown above. Fine for a single simple chart.
plt.plot(months, sales)
plt.title("...")

# Style 2: object-oriented (explicit) — verbose but essential for anything with multiple panels
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(months, sales)
ax.set_title("...")

💡 **Recommendation:** use the terse `plt.` style for one-off single charts, but switch to the explicit `fig, ax = plt.subplots()` object-oriented style the moment you need more than one panel, want to reuse plotting logic in a function, or need fine control — the "advanced" section of this chapter uses this style throughout, because it scales to complexity in a way the terse style doesn't.

### 1.2 Bar Charts — Comparing Categories

In [ ]:
regions = ["North", "South", "East", "West"]
revenue = [125, 98, 112, 145]

plt.figure(figsize=(8, 5))
bars = plt.bar(regions, revenue, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title("Regional Revenue Comparison", fontsize=14, fontweight='bold')
plt.ylabel("Revenue (₹ Lakhs)")

# Adding value labels directly on top of each bar — a small touch that makes executive charts look far more finished
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
              f'{height}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

The bar-labeling loop is worth understanding rather than just copy-pasting: `bar.get_height()` reads back the value each bar represents, `bar.get_x() + bar.get_width()/2` computes the horizontal *center* of that bar, and `plt.text(x, y, text, ha='center')` places text at that point, horizontally centered. This exact pattern — loop over the plotted objects, read their geometry, annotate — is how you'll add custom labels to almost any Matplotlib chart type.

In [ ]:
# Horizontal bars — the right choice when category names are long or there are many categories (avoids crowded x-axis labels)
products = ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset"]
units = [120, 95, 340, 850, 210]

plt.figure(figsize=(9, 5))
plt.barh(products, units, color='steelblue')
plt.title("Units Sold by Product", fontsize=14, fontweight='bold')
plt.xlabel("Units Sold")
plt.tight_layout()
plt.show()

### 1.3 Pie Charts — Use Sparingly

In [ ]:
channels = ["Online", "Retail Stores", "Distributors", "Direct Sales"]
contribution = [35, 28, 22, 15]

plt.figure(figsize=(7, 7))
plt.pie(contribution, labels=channels, autopct='%1.1f%%', startangle=90, colors=sns.color_palette("pastel"))
plt.title("Revenue Contribution by Channel", fontsize=14, fontweight='bold')
plt.axis('equal')     # forces the pie to be drawn as a circle, not an ellipse
plt.show()

⚠️ **A genuinely important piece of professional judgment, not just syntax:** pie charts are the one chart type in this whole volume you should reach for *reluctantly*. Human vision is measurably worse at comparing angles and areas than at comparing the length of bars along a common baseline — this is a well-established finding in data visualization research, not merely a stylistic preference. A bar chart of the same four channel percentages, sorted by size, communicates the same information more precisely and just as quickly. Reserve pie charts for situations where you genuinely have 2–4 categories and the *whole-to-part* relationship (not precise ranking) is the entire point — anything with 5+ slices, or where precise comparison between two similarly-sized slices matters, should be a bar chart instead.

### 1.4 Advanced Matplotlib: Subplots, Annotations, and Reusable Functions

In [ ]:
months = pd.date_range('2026-01-01', periods=6, freq='MS')     # 'MS' = month-start timestamps
sales = [120, 135, 150, 142, 168, 180]
margin = [18, 20, 21, 19, 23, 24]

fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)     # 2 rows, 1 column, sharing the x-axis
ax[0].plot(months, sales, marker='o')
ax[0].set_ylabel('Sales')
ax[1].bar(months, margin)
ax[1].set_ylabel('Margin %')
plt.tight_layout()
plt.show()

`plt.subplots(nrows, ncols)` gives you back a `fig` (the whole canvas) and an `ax` array (each individual panel — here, `ax[0]` is the top panel, `ax[1]` the bottom). `sharex=True` locks the two panels' x-axes together, so zooming or aligning one aligns both — genuinely useful when stacking related time series (sales and margin, in this example) that should visually line up month for month.

In [ ]:
# Annotating a specific point — e.g., calling out the peak month directly on the chart
best = int(np.argmax(sales))     # np.argmax returns the INDEX of the maximum value

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, sales, marker='o')
ax.annotate('Peak', xy=(months[best], sales[best]), xytext=(0.6, 0.85),
            textcoords='axes fraction', arrowprops={'arrowstyle': '->'})
plt.show()

`ax.annotate()` draws an arrow (via `arrowprops`) from a text label to a specific data point. `xy` is *where the arrow points to* (in actual data coordinates — a real date and sales value here); `xytext` combined with `textcoords='axes fraction'` places *the text itself* at a relative position within the panel (0.6, 0.85 = 60% across, 85% up), independent of the data's scale. This distinction — data coordinates for the arrow tip, axes-fraction coordinates for the label position — is what lets you place a callout label in a consistently readable spot (e.g., always in a corner) regardless of where the actual data point happens to fall.

**Building a reusable plotting function** — the professional habit that separates one-off notebook charts from a report you'll generate every month:

In [ ]:
def kpi_chart(dates, values, title, ylabel, color='teal'):
    """A reusable, consistently-styled line chart for any single KPI over time."""
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(dates, values, marker='o', linewidth=2, color=color)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    return fig, ax

kpi_chart(months, sales, "Monthly Sales", "₹ Lakhs")
kpi_chart(months, margin, "Monthly Margin", "%", color='darkorange')
plt.show()

Wrapping a chart's styling in a function (recall Volume 01's Chapter 9) means every chart in a recurring report looks identical without you re-typing (and potentially inconsistently re-typing) the same styling code every time — and if the CFO asks you to change the corporate color scheme next quarter, you edit one function instead of forty chart cells.

### 1.5 Multi-Chart Dashboards

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(months, sales, marker='o', color='teal')
axes[0, 0].set_title("Sales Trend")

axes[0, 1].bar(regions, revenue, color='coral')
axes[0, 1].set_title("Regional Revenue")

sns.boxplot(data=df, x="Region", y="Sales", ax=axes[1, 0])     # Seaborn plots accept an `ax=` too — mix freely
axes[1, 0].set_title("Sales Distribution")

axes[1, 1].pie(contribution, labels=channels, autopct='%1.0f%%')
axes[1, 1].set_title("Channel Mix")

plt.suptitle("Executive Sales Dashboard", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

For a 2×2 grid, `axes` becomes a 2D array, indexed exactly like a NumPy array (`axes[row, col]`) — the exact same indexing rules from Volume 02's NumPy chapter apply here directly. `plt.suptitle()` (note: different from `plt.title()`) sets one title for the *entire* figure, above all four panels. 💡 **Seaborn and Matplotlib compose freely** — nearly every Seaborn function accepts an `ax=` argument, letting you drop a statistical Seaborn chart into one panel of an otherwise hand-built Matplotlib dashboard, exactly as shown with the boxplot above.

### Cheat Sheet — Matplotlib

| Task | Code |
|---|---|
| Line chart | `plt.plot(x, y, marker='o')` |
| Bar chart | `plt.bar(x, y)` / horizontal: `plt.barh(x, y)` |
| Pie chart (use sparingly) | `plt.pie(values, labels=..., autopct='%1.1f%%')` |
| Multiple panels | `fig, ax = plt.subplots(rows, cols, figsize=(w,h))` |
| Annotate a point | `ax.annotate(text, xy=(x,y), xytext=(...), arrowprops={...})` |
| Whole-figure title | `plt.suptitle("...")` |
| Fix spacing | `plt.tight_layout()` |
| Save to file | `plt.savefig("chart.png", dpi=150)` |

### 🎯 Business Challenge — Matplotlib

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    "Region": np.random.choice(["North", "South", "East", "West"], 200),
    "Sales": np.random.normal(120, 30, 200)
})

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(months, sales, marker='o', color='teal')
axes[0, 0].set_title("Monthly Sales Trend")

axes[0, 1].bar(regions, revenue, color=['#1f77b4','#ff7f0e','#2ca02c','#d62728'])
axes[0, 1].set_title("Regional Comparison")

sns.boxplot(data=df, x="Region", y="Sales", ax=axes[1, 0])
axes[1, 0].set_title("Sales Distribution by Region")

sns.histplot(df["Sales"], bins=20, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Overall Sales Distribution")

plt.suptitle("Q2 Executive Dashboard", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("executive_dashboard.png", dpi=150)
plt.show()
print("Saved as executive_dashboard.png")

---